In [1]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
from backtesting import Backtest, Strategy
from backtesting.lib import crossover
from backtesting.test import SMA
from models import LSTM, Hybrid, FeedForward
import warnings
warnings.filterwarnings('ignore')

c:\Users\victo\miniconda3\envs\stock-robot\Lib\site-packages\backtesting\_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [ ]:
# Load the trained models
def load_models():
    # Initialize models
    lstm_model = LSTM()
    nn_model = FeedForward()
    hybrid_model = Hybrid()
    
    # Load the trained weights
    lstm_model.load_state_dict(torch.load('models/lstm.pth', map_location='cpu'))
    nn_model.load_state_dict(torch.load('models/feedforward.pth', map_location='cpu'))
    hybrid_model.load_state_dict(torch.load('models/hybrid.pth', map_location='cpu'))
    
    # Set to evaluation mode
    lstm_model.eval()
    nn_model.eval()
    hybrid_model.eval()
    
    return lstm_model, nn_model, hybrid_model

# Load models
lstm_model, nn_model, hybrid_model = load_models()
print("All models loaded successfully!")

All models loaded successfully!


: 

In [ ]:
# Enhanced Stock Prediction Strategy with Model Predictions
class MLPredictionStrategy(Strategy):
    """
    Trading strategy that uses ML model predictions to make buy/sell decisions
    """
    def __init__(self, model, model_name, feature_scaler=None, target_scaler=None):
        self.model = model
        self.model_name = model_name
        self.feature_scaler = feature_scaler
        self.target_scaler = target_scaler
        self.lookback = 20  # Days of data needed for features
        
    def init(self):
        # Technical indicators
        self.sma_10 = self.I(SMA, self.data.Close, 10)
        self.sma_20 = self.I(SMA, self.data.Close, 20)
        self.sma_50 = self.I(SMA, self.data.Close, 50)
        
        # RSI
        def rsi(close, period=14):
            delta = pd.Series(close).diff()
            gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
            loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
            rs = gain / loss
            return 100 - (100 / (1 + rs))
        
        self.rsi = self.I(rsi, self.data.Close, 14)
        
        # Bollinger Bands
        def bb_upper(close, period=20, std=2):
            sma = pd.Series(close).rolling(window=period).mean()
            std_dev = pd.Series(close).rolling(window=period).std()
            return sma + (std * std_dev)
        
        def bb_lower(close, period=20, std=2):
            sma = pd.Series(close).rolling(window=period).mean()
            std_dev = pd.Series(close).rolling(window=period).std()
            return sma - (std * std_dev)
        
        self.bb_upper = self.I(bb_upper, self.data.Close, 20)
        self.bb_lower = self.I(bb_lower, self.data.Close, 20)
        
        # Volume indicators
        def volume_sma(volume, period=10):
            return pd.Series(volume).rolling(window=period).mean()
        
        self.volume_sma = self.I(volume_sma, self.data.Volume, 10)
        
    def create_features(self, i):
        """Create features for model prediction at index i"""
        if i < self.lookback:
            return None
            
        # Get recent data
        close_prices = self.data.Close[i-self.lookback:i+1]
        high_prices = self.data.High[i-self.lookback:i+1]
        low_prices = self.data.Low[i-self.lookback:i+1]
        open_prices = self.data.Open[i-self.lookback:i+1]
        volumes = self.data.Volume[i-self.lookback:i+1]
        
        # Create feature array (simplified version of your features)
        features = []
        
        # Price features
        features.extend([
            close_prices[-1], high_prices[-1], low_prices[-1], open_prices[-1],
            (high_prices[-1] - low_prices[-1]) / close_prices[-1],  # Volatility
            volumes[-1] / np.mean(volumes),  # Volume ratio
        ])
        
        # Technical indicators
        sma_10_val = self.sma_10[i] if i < len(self.sma_10) else close_prices[-1]
        sma_20_val = self.sma_20[i] if i < len(self.sma_20) else close_prices[-1]
        sma_50_val = self.sma_50[i] if i < len(self.sma_50) else close_prices[-1]
        rsi_val = self.rsi[i] if i < len(self.rsi) else 50
        
        features.extend([
            sma_10_val / close_prices[-1] - 1,  # SMA ratio
            sma_20_val / close_prices[-1] - 1,
            sma_50_val / close_prices[-1] - 1,
            rsi_val / 100,  # Normalized RSI
        ])
        
        # Price momentum features
        returns = np.diff(close_prices) / close_prices[:-1]
        features.extend([
            np.mean(returns[-5:]),  # 5-day return
            np.mean(returns[-10:]),  # 10-day return
            np.std(returns[-10:]),   # 10-day volatility
            np.max(returns[-5:]),    # Max recent return
            np.min(returns[-5:]),    # Min recent return
        ])
        
        # Pad features to match model input size (48 features expected)
        while len(features) < 48:
            features.append(0.0)
        
        return np.array(features[:48], dtype=np.float32)
    
    def get_prediction(self, i):
        """Get model prediction for current timestep"""
        features = self.create_features(i)
        if features is None:
            return 0.0
            
        try:
            with torch.no_grad():
                if self.model_name == 'LSTM':
                    # For LSTM, we need sequence data
                    # Create a simple sequence by repeating features
                    features_tensor = torch.FloatTensor(features).unsqueeze(0).unsqueeze(0)
                    features_tensor = features_tensor.expand(-1, 10, -1)  # seq_len=10
                    prediction = self.model(features_tensor).item()
                elif self.model_name == 'Hybrid':
                    # For Hybrid model, use regular features
                    features_tensor = torch.FloatTensor(features).unsqueeze(0)
                    prediction = self.model(features_tensor).item()
                else:
                    # For regular NN
                    features_tensor = torch.FloatTensor(features).unsqueeze(0)
                    prediction = self.model(features_tensor).item()
                    
            return prediction
        except Exception as e:
            print(f"Prediction error: {e}")
            return 0.0
    
    def next(self):
        # Get current index
        current_idx = len(self.data) - len(self.data.Close) + len([x for x in self.data.Close if not pd.isna(x)]) - 1
        
        # Skip if not enough data
        if current_idx < self.lookback:
            return
            
        # Get model prediction
        prediction = self.get_prediction(current_idx)
        
        # Trading logic based on prediction and technical indicators
        current_price = self.data.Close[-1]
        
        # Strong buy signal: positive prediction + bullish technicals
        if (prediction > 0.02 and  # Positive prediction threshold
            self.data.Close[-1] > self.sma_10[-1] and  # Price above short-term SMA
            self.rsi[-1] < 70 and  # Not overbought
            not self.position):
            
            self.buy(size=0.95)  # Use 95% of available cash
            
        # Strong sell signal: negative prediction + bearish technicals
        elif (prediction < -0.02 and  # Negative prediction threshold
              self.data.Close[-1] < self.sma_10[-1] and  # Price below short-term SMA
              self.rsi[-1] > 30 and  # Not oversold
              self.position):
            
            self.position.close()
            
        # Stop loss and take profit
        elif self.position:
            try:
                # Use the correct attribute name for entry price
                entry_price = self.position.entry_price if hasattr(self.position, 'entry_price') else self.position.entry_bar.Close
                current_return = (current_price - entry_price) / entry_price
                
                # Stop loss at -5%
                if current_return < -0.05:
                    self.position.close()
                    
                # Take profit at +10%
                elif current_return > 0.10:
                    self.position.close()
            except (AttributeError, TypeError):
                # If we can't get entry price, just use basic risk management
                pass

# Download test data for backtesting
def get_stock_data(symbol, start_date, end_date):
    """Download stock data for backtesting"""
    try:
        data = yf.download(symbol, start=start_date, end=end_date)
        
        if isinstance(data.columns, pd.MultiIndex):
            data.columns = [col[0] for col in data.columns]

        required_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
        if all(col in data.columns for col in required_cols):
            data = data[required_cols]  # Reorder to OHLCV
        
        data = data.dropna()
        return data
    except Exception as e:
        print(f"Error downloading {symbol}: {e}")
        return None

# Backtesting configuration
INITIAL_CASH = 10000  # $10,000 starting capital
COMMISSION = 0.002    # 0.2% per transaction
START_DATE = '2023-01-01'
END_DATE = '2024-12-31'

# Test stocks (popular stocks for backtesting)
test_stocks = ['AAPL', 'GOOGL', 'MSFT', 'TSLA', 'NVDA']

print("🚀 Starting Backtesting with 3 ML Models")
print("=" * 50)
print(f"💰 Initial Capital: ${INITIAL_CASH:,}")
print(f"📅 Period: {START_DATE} to {END_DATE}")
print(f"📊 Commission: {COMMISSION*100}% per trade")
print(f"🎯 Test Stocks: {', '.join(test_stocks)}")
print("=" * 50)

# Store results
backtest_results = {}
model_names = ['FeedForward', 'LSTM', 'Hybrid']
models = [nn_model, lstm_model, hybrid_model]

for stock in test_stocks:
    print(f"\n📈 Backtesting {stock}...")
    
    # Download data
    stock_data = get_stock_data(stock, START_DATE, END_DATE)
    if stock_data is None or len(stock_data) < 100:
        print(f"❌ Insufficient data for {stock}")
        continue
    
    backtest_results[stock] = {}
    
    # Test each model
    for model, model_name in zip(models, model_names):
        print(f"  🤖 Testing {model_name} model...")
        
        try:
            # Fix: Use a factory function to properly capture variables
            def create_strategy_class(captured_model, captured_name):
                class CurrentModelStrategy(MLPredictionStrategy):
                    def __init__(self, broker, data, params):
                        # Call Strategy.__init__ first to set up data access
                        Strategy.__init__(self, broker, data, params)
                        # Then set our custom attributes
                        self.model = captured_model
                        self.model_name = captured_name
                        self.feature_scaler = None
                        self.target_scaler = None
                        self.lookback = 20
                return CurrentModelStrategy
            
            # Create the strategy class with current loop variables
            StrategyClass = create_strategy_class(model, model_name)
            
            # Run backtest
            bt = Backtest(
                stock_data, 
                StrategyClass,
                cash=INITIAL_CASH,
                commission=COMMISSION,
                exclusive_orders=True # VERY BAD (Cant have more than one position)
                # trade_on_close??
            )
            result = bt.run()
            # bt.optimize()
            
            # Store results
            backtest_results[stock][model_name] = {
                'Final Value': result['Equity Final [$]'],
                'Total Return': result['Return [%]'],
                'Sharpe Ratio': result['Sharpe Ratio'],
                'Max Drawdown': result['Max. Drawdown [%]'],
                'Total Trades': result['# Trades'],
                'Win Rate': result['Win Rate [%]'] if result['# Trades'] > 0 else 0
            }
            
            print(f"    ✅ Return: {result['Return [%]']:.2f}% | Sharpe: {result['Sharpe Ratio']:.2f} | Trades: {result['# Trades']}")
            
        except Exception as e:
            print(f"    ❌ Error with {model_name}: {e}")
            backtest_results[stock][model_name] = None
    # bt.plot()

print("\n🎉 Backtesting Complete!")
print("=" * 50)

🚀 Starting Backtesting with 3 ML Models
💰 Initial Capital: $10,000
📅 Period: 2023-01-01 to 2024-12-31
📊 Commission: 0.2% per trade
🎯 Test Stocks: AAPL, GOOGL, MSFT, TSLA, NVDA

📈 Backtesting AAPL...


[*********************100%***********************]  1 of 1 completed

  🤖 Testing FeedForward model...


Backtest.run:   0%|          | 0/451 [00:00<?, ?bar/s]

In [ ]:
# Comprehensive Backtesting Analysis with Multiple Stocks
print("🚀 COMPREHENSIVE BACKTESTING ANALYSIS")
print("="*60)

# Extended test with multiple stocks and timeframes
test_stocks = ['AAPL', 'GOOGL', 'MSFT', 'TSLA', 'V']
timeframes = [
    ('2023-01-01', '2024-06-30', '2023-2024 (18 months)'),
    ('2022-01-01', '2023-12-31', '2022-2023 (24 months)'),
]

all_results = {}

# Create visualization
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('🤖 ML Stock Trading Backtesting Results', fontsize=16, fontweight='bold')

# Performance by Model (latest period)
latest_period = list(all_results.keys())[0]
latest_data = all_results[latest_period]

models = ['FeedForward', 'LSTM', 'Hybrid']
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

# Plot 1: Returns by Model
returns_by_model = []
for model in models:
    model_returns = []
    for symbol_data in latest_data.values():
        if symbol_data and symbol_data.get(model):
            model_returns.append(symbol_data[model]['Return'])
    returns_by_model.append(model_returns)

ax1.boxplot(returns_by_model, labels=models, patch_artist=True)
for patch, color in zip(ax1.artists, colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax1.set_title('📊 Returns Distribution by Model')
ax1.set_ylabel('Return (%)')
ax1.grid(True, alpha=0.3)

# Plot 2: Stock Performance Comparison
stock_names = list(latest_data.keys())[:5]  # Top 5 stocks
x = np.arange(len(stock_names))
width = 0.25

for i, model in enumerate(models):
    model_returns = []
    for stock in stock_names:
        if latest_data[stock] and latest_data[stock].get(model):
            model_returns.append(latest_data[stock][model]['Return'])
        else:
            model_returns.append(0)
    
    ax2.bar(x + i*width, model_returns, width, label=model, color=colors[i], alpha=0.8)

ax2.set_xlabel('Stocks')
ax2.set_ylabel('Return (%)')
ax2.set_title('📈 Performance by Stock')
ax2.set_xticks(x + width)
ax2.set_xticklabels(stock_names)
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: Risk vs Return
for i, model in enumerate(models):
    returns = []
    sharpes = []
    
    for symbol_data in latest_data.values():
        if symbol_data and symbol_data.get(model):
            result = symbol_data[model]
            if result['Sharpe'] and not np.isnan(result['Sharpe']):
                returns.append(result['Return'])
                sharpes.append(result['Sharpe'])
    
    if returns and sharpes:
        ax3.scatter(sharpes, returns, label=model, color=colors[i], s=100, alpha=0.7)

ax3.set_xlabel('Sharpe Ratio')
ax3.set_ylabel('Return (%)')
ax3.set_title('⚖️ Risk-Adjusted Returns')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Plot 4: Trading Activity
trade_counts = []
model_labels = []

for model in models:
    total_trades = 0
    count = 0
    
    for symbol_data in latest_data.values():
        if symbol_data and symbol_data.get(model):
            total_trades += symbol_data[model]['Trades']
            count += 1
    
    if count > 0:
        avg_trades = total_trades / count
        trade_counts.append(avg_trades)
        model_labels.append(model)

bars = ax4.bar(model_labels, trade_counts, color=colors[:len(model_labels)], alpha=0.8)
ax4.set_title('🔄 Average Trading Activity')
ax4.set_ylabel('Average Trades per Stock')
ax4.grid(True, alpha=0.3)

# Add value labels on bars
for bar, count in zip(bars, trade_counts):
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height + 0.1,
             f'{count:.1f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

# Final Performance Summary
print("\n" + "🏆" + "="*58 + "🏆")
print("🎯 FINAL PERFORMANCE RANKING")
print("🏆" + "="*58 + "🏆")

# Calculate overall scores
model_scores = {}
for model in models:
    total_return = 0
    total_sharpe = 0
    valid_results = 0
    
    for period_data in all_results.values():
        for symbol_data in period_data.values():
            if symbol_data and symbol_data.get(model):
                result = symbol_data[model]
                if result['Sharpe'] and not np.isnan(result['Sharpe']):
                    total_return += result['Return']
                    total_sharpe += result['Sharpe']
                    valid_results += 1
    
    if valid_results > 0:
        model_scores[model] = {
            'avg_return': total_return / valid_results,
            'avg_sharpe': total_sharpe / valid_results,
            'total_tests': valid_results
        }

# Rank models
ranked_models = sorted(model_scores.items(), 
                      key=lambda x: x[1]['avg_return'], reverse=True)

for i, (model, scores) in enumerate(ranked_models, 1):
    print(f"\n🥇 Rank {i}: {model}")
    print(f"   📈 Average Return: {scores['avg_return']:+.2f}%")
    print(f"   ⚖️ Average Sharpe: {scores['avg_sharpe']:.3f}")
    print(f"   🧪 Successful Tests: {scores['total_tests']}")

print(f"\n💡 Starting with $10,000, your best strategy would have turned it into:")
for model, scores in ranked_models:
    final_value = 10000 * (1 + scores['avg_return']/100)
    profit = final_value - 10000
    print(f"   {model}: ${final_value:,.2f} ({profit:+,.2f} profit)")

print("\n" + "="*60)
print("✅ Backtesting Analysis Complete!")